In [ ]:
# ===================== SIMPLE GPT-2 RAG =====================

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

# Knowledge Base
texts = [
    "Chocolate ice cream is one of the most popular flavors.",
    "It is made using cocoa powder or melted chocolate.",
    "The base includes milk, cream, sugar, and stabilizers.",
    "Chocolate ice cream is usually high in sugar and fat.",
    "It is often used in sundaes and milkshakes.",
    "Double chocolate ice cream contains chocolate chips.",
    "Chocolate ice cream is available in dairy-free options.",
]

# Create Vector Database
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = np.array(embedding_model.encode(texts), dtype=np.float32)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# RAG Function
def ask(question):
    # Search
    q_embedding = np.array(embedding_model.encode([question]), dtype=np.float32)
    _, idx = index.search(q_embedding, k=2)

    context = " ".join([texts[i] for i in idx[0]])

    # Prompt
    prompt = f"Context: {context} Query: {question} Answer:"

    # Generate
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        inputs.input_ids,
        max_length=120,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# ================== Test karo ==================
print("GPT-2 RAG Ready! Type 'exit' to stop.\n")

while True:
    q = input("Question: ")
    if q.lower() == "exit":
        break
    ans = ask(q)
    print("Answer:", ans)
    print("-" * 50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

GPT-2 RAG Ready! Type 'exit' to stop.

Question: what is ai


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Answer: Context: Chocolate ice cream is one of the most popular flavors. Chocolate ice cream is usually high in sugar and fat. Query: what is ai Answer: i think i have it wrong. chocolate ice cream is quite good for its taste. I am not sure why. Question: if someone said chocolate ice cream tastes like ice cream then what would be their point? Answer: i guess you would say chocolate ice cream is not a good ice cream flavor. i guess you would say chocolate ice cream has this same flavor as ice cream. Question: what is the best way to make chocolate ice cream
--------------------------------------------------
Question: choclate
Answer: Context: It is made using cocoa powder or melted chocolate. Chocolate ice cream is available in dairy-free options. Query: choclate Answer: Chocolate is produced by adding cocoa powder, the same way as the chocolate ice cream. The chocolate is packed with flavor and has a dark, silky texture that is slightly sweeter than a standard chocolate ice cream. The